# AI-Based Business Location Selector
## Notebook 01: Data Cleaning, Integrity Verification & Exploratory Data Analysis (EDA)

**Domain:** Machine Learning, Geospatial Analytics & Decision Support  
**Objective:** Verify data integrity, examine distributions, analyze correlations, and identify key drivers for commercial location selection.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11


### 1. Data Ingestion
Load the candidate location dataset and inspect the schema, column types, and sample records.


In [ ]:
# Load dataset
data_path = os.path.join('..', 'data', 'business_locations.csv')
df = pd.read_csv(data_path)
print(f'Total records loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()


In [ ]:
# Summary statistics and data types
df.info()
df.describe().round(2)


### 2. Data Cleaning & Integrity Auditing
As stated in the project guide, real or synthetic data must undergo systematic validation:
1. Missing value checks
2. Duplicate identifier checks
3. Range sanity checks (e.g. rent > 0, scores within 1-10)


In [ ]:
# Check missing values and duplicate location IDs
missing_counts = df.isnull().sum()
duplicates = df.duplicated(subset=['location_id']).sum()
print('Missing Values per Column:')
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else 'None! Complete data.')
print(f'Duplicate Location IDs: {duplicates}')


In [ ]:
# Range sanity checks
assert (df['monthly_rent'] > 0).all(), 'Rent contains non-positive values'
assert (df['accessibility_score'].between(1.0, 10.0)).all(), 'Accessibility out of bounds'
assert (df['parking_score'].between(1.0, 10.0)).all(), 'Parking out of bounds'
print('All range sanity checks passed successfully!')


### 3. Univariate Distributions: Rent, Footfall, Population, and Demand
Examine continuous variables to detect skewness, multimodality, or extreme outliers.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(df['monthly_rent'], kde=True, ax=axes[0, 0], color='#2980b9')
axes[0, 0].set_title('Monthly Rent Distribution (₹)')

sns.histplot(df['daily_footfall'], kde=True, ax=axes[0, 1], color='#27ae60')
axes[0, 1].set_title('Daily Pedestrian Footfall Distribution')

sns.histplot(df['population'], kde=True, ax=axes[1, 0], color='#8e44ad')
axes[1, 0].set_title('Catchment Population Distribution')

sns.histplot(df['target_demand'], kde=True, ax=axes[1, 1], color='#e67e22')
axes[1, 1].set_title('Observed Customer Demand (Daily Transactions)')
plt.tight_layout()
plt.show()


### 4. Categorical Feature Distributions
Analyze candidate representation across commercial industries and purchasing power segments.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, y='business_type', ax=axes[0], palette='Blues_r', order=df['business_type'].value_counts().index)
axes[0].set_title('Candidate Locations per Business Category')

sns.countplot(data=df, x='income_level', ax=axes[1], palette='Purples', order=['Low', 'Medium', 'High'])
axes[1].set_title('Income / Purchasing Power Distribution')
plt.tight_layout()
plt.show()


### 5. Correlation Heatmap
Evaluate collinearity and dependencies among numeric variables to inform feature selection.


In [ ]:
numeric_cols = [
    'monthly_rent', 'population', 'competitor_count', 'daily_footfall',
    'accessibility_score', 'parking_score', 'public_transport_score', 'target_demand'
]
corr_matrix = df[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Commercial & Geospatial Factors')
plt.show()


### 6. Bivariate Analysis: Rent vs. Footfall by Business Potential
Observe whether premium rent areas correspond with higher customer footfall and business viability.


In [ ]:
plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=df,
    x='monthly_rent',
    y='target_demand',
    hue='business_potential',
    size='daily_footfall',
    sizes=(30, 250),
    palette='Set2',
    alpha=0.85
)
plt.title('Rent vs. Customer Demand Segregated by Business Potential')
plt.xlabel('Monthly Rent (₹)')
plt.ylabel('Daily Customer Demand')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


### 7. Key Findings & EDA Summary
1. **Footfall & Accessibility Drive Demand:** Pedestrian volume and transit composite show the strongest direct positive association with customer transactions.
2. **Rent vs. Viability Trade-off:** High rent does not guarantee high potential; the sweet spot lies in locations with moderate rent and high pedestrian access.
3. **Competition Dynamics:** Moderate competition signifies a proven customer base, but beyond an optimal saturation threshold, returns diminish.
